# MLG382 Guided Project — BC Analytics
## Diabetes Risk Segmentation & Decision Support System


## 1. Imports & Configuration

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import (
    LabelEncoder, OrdinalEncoder, StandardScaler, MinMaxScaler, RobustScaler
)
from sklearn.model_selection import train_test_split

import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)

RANDOM_STATE = 42
print('Libraries loaded successfully.')

---
## 2. Load Data

In [ ]:
df = pd.read_csv('Diabetes_and_LifeStyle_Dataset_.csv')

print(f'Dataset shape: {df.shape}')
print(f'Rows: {df.shape[0]:,} | Columns: {df.shape[1]}')
df.head()

In [ ]:
print('Column names and data types:')
print(df.dtypes)

---
## 3. Domain Understanding — Diabetes Stage (Target Variable)

Before any preprocessing, we must understand what each class in `diabetes_stage` represents clinically:



In [ ]:
print('Target class distribution:')
stage_counts = df['diabetes_stage'].value_counts()
print(stage_counts)
print(f'\nClass imbalance ratio (largest/smallest): {stage_counts.max() / stage_counts.min():.1f}x')

fig, ax = plt.subplots(figsize=(8, 4))
stage_counts.plot(kind='bar', ax=ax, color=['#2ECC71','#F39C12','#E74C3C','#3498DB','#9B59B6'], edgecolor='black')
ax.set_title('Distribution of Diabetes Stage (Target Variable)', fontweight='bold')
ax.set_xlabel('Diabetes Stage')
ax.set_ylabel('Count')
ax.tick_params(axis='x', rotation=15)
for p in ax.patches:
    ax.annotate(f'{int(p.get_height()):,}', (p.get_x() + p.get_width()/2, p.get_height()),
                ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.show()

---
## 4. Missing Value Analysis

In [ ]:
missing = df.isnull().sum()
missing_pct = (missing / len(df)) * 100
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
missing_df = missing_df[missing_df['Missing Count'] > 0]

if missing_df.empty:
    print('No missing values detected in the dataset.')
else:
    print('Missing values found:')
    print(missing_df)

---
## 5. Feature Categorisation

We split the 31 columns into logical groups before applying transformations.

In [ ]:
# ----- Feature groups -----

TARGET = 'diabetes_stage'


NOMINAL_COLS = ['gender', 'ethnicity', 'employment_status', 'smoking_status']


ORDINAL_COLS = ['education_level', 'income_level']


BINARY_COLS = ['family_history_diabetes', 'hypertension_history',
               'cardiovascular_history', 'diagnosed_diabetes']

NUMERICAL_COLS = [
    'Age', 'alcohol_consumption_per_week', 'physical_activity_minutes_per_week',
    'diet_score', 'sleep_hours_per_day', 'screen_time_hours_per_day',
    'bmi', 'waist_to_hip_ratio', 'systolic_bp', 'diastolic_bp',
    'heart_rate', 'cholesterol_total', 'hdl_cholesterol', 'ldl_cholesterol',
    'triglycerides', 'glucose_fasting', 'glucose_postprandial',
    'insulin_level', 'hba1c', 'diabetes_risk_score'
]

print(f'Nominal features   : {len(NOMINAL_COLS)} — {NOMINAL_COLS}')
print(f'Ordinal features   : {len(ORDINAL_COLS)} — {ORDINAL_COLS}')
print(f'Binary features    : {len(BINARY_COLS)} — {BINARY_COLS}')
print(f'Numerical features : {len(NUMERICAL_COLS)}')
print(f'Target             : {TARGET}')

---
## 6. Duplicate Check

In [ ]:
n_dupes = df.duplicated().sum()
print(f'Duplicate rows: {n_dupes}')

if n_dupes > 0:
    df = df.drop_duplicates().reset_index(drop=True)
    print(f'Duplicates removed. New shape: {df.shape}')
else:
    print('No duplicate rows — dataset is clean.')

---
## 7. Outlier Detection & Treatment (IQR Method)

For clinical/health data, extreme outliers can represent data entry errors. We use **IQR capping (Winsorisation)** — clipping values at the 1st and 99th percentiles — to retain all rows while reducing the distorting effect of extreme values.

In [ ]:
def detect_outliers_iqr(dataframe, columns):
    """Return a summary of outliers using the IQR method."""
    records = []
    for col in columns:
        Q1 = dataframe[col].quantile(0.25)
        Q3 = dataframe[col].quantile(0.75)
        IQR = Q3 - Q1
        lower = Q1 - 1.5 * IQR
        upper = Q3 + 1.5 * IQR
        n_out = ((dataframe[col] < lower) | (dataframe[col] > upper)).sum()
        records.append({'feature': col, 'lower_bound': lower, 'upper_bound': upper,
                        'outlier_count': n_out, 'outlier_%': round(n_out / len(dataframe) * 100, 2)})
    return pd.DataFrame(records).set_index('feature')


outlier_summary = detect_outliers_iqr(df, NUMERICAL_COLS)
print('Outlier summary (IQR method):')
outlier_summary[outlier_summary['outlier_count'] > 0]

In [ ]:

df_clean = df.copy()

for col in NUMERICAL_COLS:
    p01 = df_clean[col].quantile(0.01)
    p99 = df_clean[col].quantile(0.99)
    df_clean[col] = df_clean[col].clip(lower=p01, upper=p99)

print('Winsorisation applied (1st–99th percentile clipping).')

check_cols = ['insulin_level', 'triglycerides', 'glucose_postprandial', 'bmi']
fig, axes = plt.subplots(2, 4, figsize=(16, 6))

for i, col in enumerate(check_cols):
    axes[0, i].hist(df[col], bins=50, color='#E74C3C', edgecolor='none', alpha=0.7)
    axes[0, i].set_title(f'{col}\n(Before)', fontsize=9)
    axes[1, i].hist(df_clean[col], bins=50, color='#2ECC71', edgecolor='none', alpha=0.7)
    axes[1, i].set_title(f'{col}\n(After Winsorisation)', fontsize=9)

plt.suptitle('Outlier Treatment — Before vs After Winsorisation', fontweight='bold')
plt.tight_layout()
plt.show()

---
## 8. Encoding Categorical Variables

### 8.1 Ordinal Encoding — `education_level` & `income_level`

In [ ]:

education_order = ['No formal', 'Highschool', 'Graduate', 'Postgraduate']
income_order    = ['Low', 'Lower-Middle', 'Middle', 'Upper-Middle', 'High']

ordinal_enc = OrdinalEncoder(
    categories=[education_order, income_order],
    handle_unknown='use_encoded_value',
    unknown_value=-1
)

df_clean[['education_level_enc', 'income_level_enc']] = ordinal_enc.fit_transform(
    df_clean[ORDINAL_COLS]
)

print('Education level mapping:', dict(zip(education_order, range(len(education_order)))))
print('Income level mapping   :', dict(zip(income_order, range(len(income_order)))))
df_clean[['education_level', 'education_level_enc', 'income_level', 'income_level_enc']].head()

### 8.2 One-Hot Encoding — Nominal Categoricals

In [ ]:
df_clean = pd.get_dummies(
    df_clean,
    columns=NOMINAL_COLS,
    drop_first=True,     
    dtype=int
)


new_dummies = [c for c in df_clean.columns if any(c.startswith(n+'_') for n in NOMINAL_COLS)]
print(f'One-hot encoded columns ({len(new_dummies)}): {new_dummies}')

### 8.3 Target Encoding — `diabetes_stage`

We create two versions of the target:
- **Label-encoded integer** — for tree-based classifiers (DT, RF, XGBoost)
- **Original string** — preserved for interpretability

In [ ]:
label_enc = LabelEncoder()
df_clean['diabetes_stage_enc'] = label_enc.fit_transform(df_clean['diabetes_stage'])

stage_mapping = dict(zip(label_enc.classes_, label_enc.transform(label_enc.classes_)))
print('Target label mapping:', stage_mapping)

df_clean[['diabetes_stage', 'diabetes_stage_enc']].value_counts().reset_index().sort_values('diabetes_stage_enc')

---
## 9. Feature Engineering

We create clinically meaningful derived features to capture interactions and risk factors that individual variables cannot express alone.

In [ ]:

def bmi_category(bmi):
    if bmi < 18.5: return 0   
    elif bmi < 25: return 1   
    elif bmi < 30: return 2   
    else:          return 3   

df_clean['bmi_category'] = df_clean['bmi'].apply(bmi_category)
print('BMI categories: 0=Underweight | 1=Normal | 2=Overweight | 3=Obese')
print(df_clean['bmi_category'].value_counts().sort_index())

def glucose_category(g):
    if g < 100: return 0  
    elif g < 126: return 1  
    else: return 2          

df_clean['glucose_fasting_category'] = df_clean['glucose_fasting'].apply(glucose_category)
print('\nFasting glucose categories: 0=Normal | 1=Pre-diabetic | 2=Diabetic range')
print(df_clean['glucose_fasting_category'].value_counts().sort_index())


def hba1c_category(h):
    if h < 5.7: return 0    
    elif h < 6.5: return 1  
    else: return 2         

df_clean['hba1c_category'] = df_clean['hba1c'].apply(hba1c_category)
print('\nHbA1c categories: 0=Normal | 1=Pre-diabetic | 2=Diabetic')
print(df_clean['hba1c_category'].value_counts().sort_index())

In [ ]:

def bp_category(systolic, diastolic):
    if systolic < 120 and diastolic < 80:  return 0  
    elif systolic < 130 and diastolic < 80: return 1  
    elif systolic < 140 or diastolic < 90:  return 2  
    else: return 3                                     

df_clean['bp_category'] = df_clean.apply(
    lambda row: bp_category(row['systolic_bp'], row['diastolic_bp']), axis=1
)
print('BP categories: 0=Normal | 1=Elevated | 2=HTN Stage 1 | 3=HTN Stage 2')
print(df_clean['bp_category'].value_counts().sort_index())

df_clean['pulse_pressure'] = df_clean['systolic_bp'] - df_clean['diastolic_bp']
print('\nPulse pressure — sample stats:')
print(df_clean['pulse_pressure'].describe())

In [ ]:

df_clean['cholesterol_ratio'] = df_clean['cholesterol_total'] / df_clean['hdl_cholesterol'].replace(0, np.nan)
df_clean['cholesterol_ratio'] = df_clean['cholesterol_ratio'].fillna(df_clean['cholesterol_ratio'].median())


df_clean['ldl_hdl_ratio'] = df_clean['ldl_cholesterol'] / df_clean['hdl_cholesterol'].replace(0, np.nan)
df_clean['ldl_hdl_ratio'] = df_clean['ldl_hdl_ratio'].fillna(df_clean['ldl_hdl_ratio'].median())

print('Cholesterol ratio stats:')
print(df_clean['cholesterol_ratio'].describe())

print('\nLDL/HDL ratio stats:')
print(df_clean['ldl_hdl_ratio'].describe())

In [ ]:


df_clean['smoking_risk']   = df_clean.get('smoking_status_Former', df_clean.get('smoking_status_Never', pd.Series(0, index=df_clean.index)))


smoking_cols = [c for c in df_clean.columns if 'smoking_status' in c]
print('Smoking dummy columns:', smoking_cols)


df_clean['low_activity_flag']  = (df_clean['physical_activity_minutes_per_week'] < 30).astype(int)  
df_clean['poor_diet_flag']     = (df_clean['diet_score'] < 4).astype(int)      
df_clean['high_alcohol_flag']  = (df_clean['alcohol_consumption_per_week'] > 7).astype(int)  
df_clean['poor_sleep_flag']    = ((df_clean['sleep_hours_per_day'] < 6) | (df_clean['sleep_hours_per_day'] > 9)).astype(int)
df_clean['high_screen_flag']   = (df_clean['screen_time_hours_per_day'] > 8).astype(int)     

lifestyle_flags = ['low_activity_flag','poor_diet_flag','high_alcohol_flag','poor_sleep_flag','high_screen_flag']
df_clean['lifestyle_risk_score'] = df_clean[lifestyle_flags].sum(axis=1)

print('\nLifestyle risk score distribution:')
print(df_clean['lifestyle_risk_score'].value_counts().sort_index())

In [ ]:

age_bins   = [0, 29, 44, 59, 74, 100]
age_labels = [0, 1, 2, 3, 4]         
df_clean['age_group'] = pd.cut(df_clean['Age'], bins=age_bins, labels=age_labels, right=True).astype(int)

print('Age group distribution:')
age_label_names = {0:'Young Adult (≤29)',1:'Adult (30–44)',2:'Middle-Aged (45–59)',3:'Senior (60–74)',4:'Elderly (75+)'}
print(df_clean['age_group'].map(age_label_names).value_counts())



df_clean['high_triglycerides'] = (df_clean['triglycerides'] > 150).astype(int)
df_clean['low_hdl']            = (df_clean['hdl_cholesterol'] < 40).astype(int)
df_clean['high_glucose']       = (df_clean['glucose_fasting'] > 100).astype(int)
df_clean['high_bp_flag']       = ((df_clean['systolic_bp'] > 130) | (df_clean['diastolic_bp'] > 85)).astype(int)
df_clean['high_whr']           = (df_clean['waist_to_hip_ratio'] > 0.90).astype(int)

metabolic_flags = ['high_triglycerides','low_hdl','high_glucose','high_bp_flag','high_whr']
df_clean['metabolic_syndrome_score'] = df_clean[metabolic_flags].sum(axis=1)
df_clean['metabolic_syndrome_flag']  = (df_clean['metabolic_syndrome_score'] >= 3).astype(int)

print('\nMetabolic syndrome flag:')
print(df_clean['metabolic_syndrome_flag'].value_counts())

In [ ]:


df_clean['glucose_fasting_mmol'] = df_clean['glucose_fasting'] / 18.01
df_clean['homa_ir'] = (df_clean['glucose_fasting_mmol'] * df_clean['insulin_level']) / 22.5
df_clean.drop(columns=['glucose_fasting_mmol'], inplace=True)

print('HOMA-IR (insulin resistance proxy) stats:')
print(df_clean['homa_ir'].describe())
print('\n  HOMA-IR > 2.5 is generally considered insulin resistant.')
print(f"Patients with HOMA-IR > 2.5: {(df_clean['homa_ir'] > 2.5).sum():,}")

In [ ]:

df_clean['glucose_spike'] = df_clean['glucose_postprandial'] - df_clean['glucose_fasting']

print('Glucose spike stats:')
print(df_clean['glucose_spike'].describe())


df_clean['comorbidity_count'] = (
    df_clean['hypertension_history'] +
    df_clean['cardiovascular_history'] +
    df_clean['family_history_diabetes']
)
print('\nComorbidity count distribution:')
print(df_clean['comorbidity_count'].value_counts().sort_index())

In [ ]:
\
engineered_features = [
    'bmi_category', 'glucose_fasting_category', 'hba1c_category', 'bp_category',
    'pulse_pressure', 'cholesterol_ratio', 'ldl_hdl_ratio',
    'low_activity_flag', 'poor_diet_flag', 'high_alcohol_flag', 'poor_sleep_flag',
    'high_screen_flag', 'lifestyle_risk_score',
    'age_group', 'metabolic_syndrome_score', 'metabolic_syndrome_flag',
    'homa_ir', 'glucose_spike', 'comorbidity_count'
]

print(f'Total engineered features created: {len(engineered_features)}')
print('\nSample of engineered features:')
df_clean[engineered_features].head(5)

---
## 10. Feature Scaling

We apply **two scaling strategies** to support different downstream models:


In [ ]:

CONTINUOUS_FEATURES = NUMERICAL_COLS + [
    'pulse_pressure', 'cholesterol_ratio', 'ldl_hdl_ratio',
    'homa_ir', 'glucose_spike'
]

print(f'Scaling {len(CONTINUOUS_FEATURES)} continuous features...')

std_scaler = StandardScaler()
std_scaled = std_scaler.fit_transform(df_clean[CONTINUOUS_FEATURES])
df_std_scaled = pd.DataFrame(std_scaled, columns=[f'{c}_std' for c in CONTINUOUS_FEATURES])


rob_scaler = RobustScaler()
rob_scaled = rob_scaler.fit_transform(df_clean[CONTINUOUS_FEATURES])
df_rob_scaled = pd.DataFrame(rob_scaled, columns=[f'{c}_rob' for c in CONTINUOUS_FEATURES])

print('StandardScaler applied (suffix: _std)')
print('RobustScaler applied (suffix: _rob)')


print('\nStandardScaler check (should be ~0 mean, ~1 std):')
print(df_std_scaled.describe().T[['mean','std']].head(5))

In [ ]:

sample_cols = ['bmi', 'hba1c', 'glucose_fasting', 'homa_ir']
fig, axes = plt.subplots(3, 4, figsize=(18, 10))

for i, col in enumerate(sample_cols):

    axes[0, i].hist(df_clean[col], bins=50, color='#3498DB', edgecolor='none', alpha=0.8)
    axes[0, i].set_title(f'{col}\nOriginal', fontsize=9)

   
    axes[1, i].hist(df_std_scaled[f'{col}_std'], bins=50, color='#E67E22', edgecolor='none', alpha=0.8)
    axes[1, i].set_title(f'{col}\nStandardScaler', fontsize=9)


    axes[2, i].hist(df_rob_scaled[f'{col}_rob'], bins=50, color='#27AE60', edgecolor='none', alpha=0.8)
    axes[2, i].set_title(f'{col}\nRobustScaler', fontsize=9)

plt.suptitle('Feature Scaling Comparison — Original vs StandardScaler vs RobustScaler', fontweight='bold')
plt.tight_layout()
plt.show()

---
## 11. Build Final Modelling Datasets

We construct **two clean final dataframes** for modelling:

- `df_classification` — for Decision Tree / Random Forest / XGBoost (uses StandardScaler features)
- `df_clustering` — for K-Means (uses StandardScaler features only, excludes target)

In [ ]:

ohe_cols = [c for c in df_clean.columns if any(c.startswith(n+'_') for n in NOMINAL_COLS)]


categorical_features = (
    ['education_level_enc', 'income_level_enc'] +
    BINARY_COLS +
    ohe_cols +
    ['bmi_category','glucose_fasting_category','hba1c_category','bp_category',
     'low_activity_flag','poor_diet_flag','high_alcohol_flag','poor_sleep_flag',
     'high_screen_flag','lifestyle_risk_score','age_group',
     'metabolic_syndrome_score','metabolic_syndrome_flag',
     'comorbidity_count','high_triglycerides','low_hdl','high_glucose','high_bp_flag','high_whr']
)


df_final = pd.concat([
    df_std_scaled.reset_index(drop=True),
    df_clean[categorical_features].reset_index(drop=True),
    df_clean[['diabetes_stage', 'diabetes_stage_enc']].reset_index(drop=True)
], axis=1)


print('NOTE: `diagnosed_diabetes` may be a leakage risk for classification.')
print('    Consider removing it if it directly determines diabetes_stage.')
print(f'\nFinal dataset shape: {df_final.shape}')
df_final.head(3)

In [ ]:

LEAKAGE_COLS = ['diagnosed_diabetes', 'diabetes_stage']  

feature_cols = [c for c in df_final.columns if c not in LEAKAGE_COLS + ['diabetes_stage_enc']]

X = df_final[feature_cols]
y = df_final['diabetes_stage_enc']

print(f'Features (X) shape : {X.shape}')
print(f'Target (y) shape   : {y.shape}')
print(f'Class distribution :\n{y.value_counts()}')

In [ ]:

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y          
)

print(f'Training set   : {X_train.shape}')
print(f'Test set       : {X_test.shape}')
print(f'\nTraining class distribution:')
print(y_train.value_counts())
print(f'\nTest class distribution:')
print(y_test.value_counts())

In [ ]:

CLUSTERING_FEATURES = (
    [f'{c}_std' for c in [
        'Age', 'bmi', 'waist_to_hip_ratio', 'systolic_bp', 'diastolic_bp',
        'glucose_fasting', 'glucose_postprandial', 'hba1c', 'insulin_level',
        'physical_activity_minutes_per_week', 'diet_score', 'sleep_hours_per_day',
        'screen_time_hours_per_day', 'cholesterol_total', 'hdl_cholesterol',
        'triglycerides', 'diabetes_risk_score', 'homa_ir'
    ]] +
    ['bmi_category', 'lifestyle_risk_score', 'metabolic_syndrome_score',
     'comorbidity_count', 'age_group']
)

X_cluster = df_final[CLUSTERING_FEATURES]
print(f'Clustering dataset shape: {X_cluster.shape}')
print(f'\nFeatures used for K-Means clustering ({len(CLUSTERING_FEATURES)}):')
print(CLUSTERING_FEATURES)

---
## 12. Correlation Analysis

In [ ]:

corr_with_target = df_final.drop(columns=['diabetes_stage'], errors='ignore').corr()['diabetes_stage_enc'].drop('diabetes_stage_enc')
top_corr = corr_with_target.abs().sort_values(ascending=False).head(20)

fig, ax = plt.subplots(figsize=(10, 6))
top_corr.plot(kind='barh', ax=ax, color='#2980B9')
ax.set_xlabel('Absolute Correlation with diabetes_stage_enc')
ax.set_title('Top 20 Features Correlated with Diabetes Stage', fontweight='bold')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

print('Top 10 most correlated features:')
print(corr_with_target.abs().sort_values(ascending=False).head(10))

In [ ]:

key_features = [
    'Age_std', 'bmi_std', 'hba1c_std', 'glucose_fasting_std',
    'glucose_postprandial_std', 'insulin_level_std', 'homa_ir_std',
    'diabetes_risk_score_std', 'cholesterol_total_std', 'triglycerides_std',
    'systolic_bp_std', 'metabolic_syndrome_score', 'lifestyle_risk_score',
    'diabetes_stage_enc'
]

corr_matrix = df_final[key_features].corr()

plt.figure(figsize=(12, 9))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm',
            linewidths=0.5, annot_kws={'size': 7})
plt.title('Correlation Heatmap — Key Clinical & Lifestyle Features', fontweight='bold')
plt.tight_layout()
plt.show()

---
## 13. Class Imbalance Assessment

The dataset has severe class imbalance (Type 1 = 117 samples; Type 2 = 58,163). This must be addressed before training classification models.

In [ ]:
print('Class imbalance summary:')
print(y_train.value_counts())
print(f'\nImbalance ratio: {y_train.value_counts().max() / y_train.value_counts().min():.1f}x')
print("""
Recommended strategies (to be applied in modelling notebook):
  1. class_weight='balanced' in Decision Tree & Random Forest
  2. scale_pos_weight in XGBoost
  3. SMOTE oversampling for minority classes (Type 1, Gestational)
  4. Use macro F1-score / weighted recall as primary metric (not accuracy)
""")

---
## 14. Save Preprocessed Datasets

In [ ]:

X_train.to_csv('X_train.csv', index=False)
X_test.to_csv('X_test.csv', index=False)
y_train.to_csv('y_train.csv', index=False)
y_test.to_csv('y_test.csv', index=False)
X_cluster.to_csv('X_cluster.csv', index=False)
df_final.to_csv('df_preprocessed.csv', index=False)


import json
with open('label_mapping.json', 'w') as f:
    json.dump(stage_mapping, f, indent=2)

print('  Files saved:')
print('  X_train.csv          — training features')
print('  X_test.csv           — test features')
print('  y_train.csv          — training labels')
print('  y_test.csv           — test labels')
print('  X_cluster.csv        — clustering features (K-Means)')
print('  df_preprocessed.csv  — full preprocessed dataset')
print('  label_mapping.json   — target class encoding map')